# Step 1: Data Profiling & Structural Audit

**E-Commerce 360° Business Intelligence Platform**

Comprehensive inspection and audit of raw Olist tables.


### 1. Environment & Setup
Import essential libraries and inspect the raw Olist e-commerce datasets.

In [1]:
import os
import glob
import pandas as pd
import numpy as np

RAW_DATA_DIR = r'data/raw'
print('Listing files in raw directory:')
for f in sorted(glob.glob(f'{RAW_DATA_DIR}/*.csv')):
    print(' -', os.path.basename(f))

Listing files in raw directory:
 - olist_customers_dataset.csv
 - olist_geolocation_dataset.csv
 - olist_order_items_dataset.csv
 - olist_order_payments_dataset.csv
 - olist_order_reviews_dataset.csv
 - olist_orders_dataset.csv
 - olist_products_dataset.csv
 - olist_sellers_dataset.csv
 - product_category_name_translation.csv


### 2. Full Structural Audit & Missing Value Assessment
Iterate through all 9 CSV files, calculating row counts, columns, memory footprint, and column-wise null percentages.

In [2]:
files = sorted(glob.glob(f'{RAW_DATA_DIR}/*.csv'))
summary = []

for f in files:
    filename = os.path.basename(f)
    df = pd.read_csv(f)
    rows, cols = df.shape
    missing_count = df.isnull().sum().sum()
    dup_count = df.duplicated().sum()
    memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    
    summary.append({
        'Dataset': filename,
        'Rows': rows,
        'Cols': cols,
        'Total Nulls': missing_count,
        'Duplicates': dup_count,
        'Memory (MB)': round(memory_mb, 2)
    })

summary_df = pd.DataFrame(summary)
summary_df

,Dataset,Rows,Cols,Total Nulls,Duplicates,Memory (MB)
0,olist_customers_dataset.csv,99441,5,0,0,26.59
1,olist_geolocation_dataset.csv,1000163,5,0,261831,130.26
2,olist_order_items_dataset.csv,112650,7,0,0,35.99
3,olist_order_payments_dataset.csv,103886,5,0,0,16.23
4,olist_order_reviews_dataset.csv,99224,7,145903,0,39.12
5,olist_orders_dataset.csv,99441,8,4908,0,52.94
6,olist_products_dataset.csv,32951,9,2448,0,6.30
7,olist_sellers_dataset.csv,3095,4,0,0,0.59
8,product_category_name_translation.csv,71,2,0,0,0.01


### 3. Orders Timestamp Completeness Audit
Inspect delivery status and timestamp progression across order lifecycles.

In [3]:
df_orders = pd.read_csv(f'{RAW_DATA_DIR}/olist_orders_dataset.csv')
print('Order Status Distribution:')
print(df_orders['order_status'].value_counts())

print('\nTimestamp Null Percentages:')
for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    print(f' - {col}: {df_orders[col].isnull().mean():.2%}')

Order Status Distribution:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Timestamp Null Percentages:
 - order_purchase_timestamp: 0.00%
 - order_approved_at: 0.16%
 - order_delivered_carrier_date: 1.79%
 - order_delivered_customer_date: 2.98%
 - order_estimated_delivery_date: 0.00%


### 4. Summary & Observations
- Over 99.4k orders span across 2016-2018 with 97% delivery rate.
- Customer ID vs Customer Unique ID distinction is critical for retention metrics.
- Geolocation dataset contains duplicate prefixes due to coordinate variations, requiring prefix aggregation.